# Counting Dynamic NIAH analysis

This Colab notebook generates and analyzes the `match_count` and `literal_count` Dynamic NIAH v2 tasks. `match_count` counts inserted city-score facts; `literal_count` counts exact copies of a per-example canary string. Like `single-example-v2.ipynb`, runtime artifacts are written under `/content/{RUN_NAME}` and the final zip is copied to `RESULTS_PATH`. The notebook keeps Google Drive traffic limited to model cache reads/writes and the final archive.


## 1. Mount Drive and choose the repo

Set `REPO_DIR` to your checked-out `dataset-generation` repository. If you cloned into `/content`, point `REPO_DIR` there instead of Drive. All commands below run from `REPO_DIR`, while generated run files stay under `/content/{RUN_NAME}`.


In [2]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

# CHANGE THIS to your checked-out dataset-generation repository.
REPO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v10')
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f'REPO_DIR does not exist: {REPO_DIR}. '
        'Update REPO_DIR to your dataset-generation checkout before continuing.'
    )

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

from counting import (
    build_counting_run_name,
    cleanup_counting_archive_artifacts,
    format_counting_cleanup_summary,
    run_counting_ablation_examples,
    validate_selected_example_ids,
)
from dataset_generation.run_utils import archive_directory
from single_example import (
    DEFAULT_ABLATION_CONFIG_PATH,
    DEFAULT_REPRESENTATION_ABLATION_CONFIG_PATH,
    DEFAULT_REPRESENTATION_RESTORE_CONFIG_PATH,
)


def run_streamed(cmd, *, cwd=REPO_DIR):
    """Run a command and stream stdout/stderr live in notebook output."""
    cmd = [str(part) for part in cmd]
    env = {**os.environ, 'PYTHONPATH': 'src', 'PYTHONUNBUFFERED': '1'}
    print('$ ' + shlex.join(cmd), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    return_code = proc.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)


print('Working directory:', Path.cwd())


Mounted at /content/drive
Working directory: /content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v10


## 2. Global configuration

For hidden-state analysis, keep exactly one `True` value in `CONTROL_SWITCH`, and make sure that controlled needle has a non-null insertion position. `USER_RUN_NAME = None` creates one timestamped `RUN_NAME` up front so dataset generation, optional response generation, hidden-state analysis, ablation analysis, and final archiving all read/write the same `/content/{RUN_NAME}` directory. Ablation analysis loops over `SELECT_EXAMPLE_ID` and writes one subfolder per selected example under `/content/{RUN_NAME}/ablation_examples/`.

Set `ANALYZE_REASONING_TOKENS=True` with `USE_THINKING=True` to generate Qwen thinking tokens, save `prompt + reasoning-before-final-answer` tensors, append `_cot` to the run name, and run hidden-state/ablation analysis on the extended inputs.


In [3]:
TASK_TYPE = 'match_count'  # 'match_count' or 'literal_count'
USER_RUN_NAME = None  # Use a string to force a stable rerun folder name.
RUN_ROOT = Path('/content')
RUN_GENERATION_EVAL = True
USE_THINKING = False  # Overrides the thinking-mode setting for generation/eval and CoT analysis.
# If True, generate reasoning tokens first, save prompt+reasoning as extended inputs,
# and run hidden-state/ablation analysis on those extended inputs. Requires USE_THINKING=True.
ANALYZE_REASONING_TOKENS = False
MAX_NEW_TOKENS_FOR_COT = 64  # Short final-answer budget when scoring CoT ablation baselines/inputs.
RUN_HIDDEN_STATE_ANALYSIS = True
DELETE_LARGE_PT_WHEN_DONE = True
# Final archive destination. If None, this is set after RUN_NAME is known.
RESULTS_PATH = None

MODEL_NAME = 'Qwen/Qwen3-8B'
TOKENIZER_NAME = MODEL_NAME
NUM_EXAMPLES = 20
TARGET_HAYSTACK_TOKENS = 1000
NUM_NEEDLES = 3
INSERTION_POSITIONS = [100, 200, 400]  # Use None/null to generate but skip an aligned needle.
CONTROL_SWITCH = [True, False, False]  # Hidden-state and ablation analyses expect exactly one True.
LAYERS = [4, 8, 12, 16, 20, 24, 28]
PCA_TEST_COUNT = NUM_EXAMPLES // 2  # Hidden-state PCA plots use the first half as test examples.
GLOBAL_RANDOM_SEED = 42
HAYSTACK_SEED = 123
NEEDLE_SEED = 456
PROMPT_STYLE = 'easier'

# Ablation example selection. The list must be a subset of list(range(NUM_EXAMPLES // 2)).
# Examples [0, 1, 2] write to /content/{RUN_NAME}/ablation_examples/example_id_0, etc.
SELECT_EXAMPLE_ID = [1, 5, 9]

# Token-level ablation settings. Defaults come from configs/ablation.json; set these to override in Colab.
RUN_ABLATION = True
ABLATION_CONFIG_PATH = DEFAULT_ABLATION_CONFIG_PATH
NUM_CRITICAL_TOKENS = None
ABLATION_RANDOM_SEED = None
CRITICAL_TOKEN_CALC_LAYER = None

# Representation-level ablation settings. Defaults come from configs/ablation-representation.json.
RUN_REPRESENTATION_ABLATION = True
ABLATION_REPRESENTATION_CONFIG_PATH = DEFAULT_REPRESENTATION_ABLATION_CONFIG_PATH
REPRESENTATION_NUM_CRITICAL_TOKENS = None
RANDOMIZE_FROM_TOP_LAYER = None

# Representation-level restore settings. Defaults come from configs/ablation-representation-restore.json.
RUN_REPRESENTATION_RESTORE = True
ABLATION_REPRESENTATION_RESTORE_CONFIG_PATH = DEFAULT_REPRESENTATION_RESTORE_CONFIG_PATH
RESTORE_NUM_CRITICAL_TOKENS = None
RESTORE_RANDOMIZE_FROM_TOP_LAYER = None

# Keep Hugging Face model downloads on Drive so reconnects do not redownload everything.
HF_CACHE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/huggingface_models')
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)

assert TASK_TYPE in {'match_count', 'literal_count'}
assert len(INSERTION_POSITIONS) == NUM_NEEDLES
assert len(CONTROL_SWITCH) == NUM_NEEDLES
assert all(isinstance(layer, int) and layer >= 0 for layer in LAYERS)
if ANALYZE_REASONING_TOKENS and not USE_THINKING:
    raise ValueError('ANALYZE_REASONING_TOKENS=True requires USE_THINKING=True.')
SELECT_EXAMPLE_ID = validate_selected_example_ids(SELECT_EXAMPLE_ID, num_examples=NUM_EXAMPLES)
if RUN_HIDDEN_STATE_ANALYSIS or RUN_ABLATION or RUN_REPRESENTATION_ABLATION or RUN_REPRESENTATION_RESTORE:
    assert sum(bool(x) for x in CONTROL_SWITCH) == 1, 'Hidden-state/ablation analysis expects exactly one control needle.'
    ctrl_idx = next(i for i, x in enumerate(CONTROL_SWITCH) if x)
    assert INSERTION_POSITIONS[ctrl_idx] is not None, 'The controlled needle must be inserted for hidden-state/ablation analysis.'

RUN_NAME = USER_RUN_NAME or build_counting_run_name(
    model_name=MODEL_NAME,
    task_type=TASK_TYPE,
    prompt_style=PROMPT_STYLE,
    target_haystack_tokens=TARGET_HAYSTACK_TOKENS,
    insertion_positions=INSERTION_POSITIONS,
)
if ANALYZE_REASONING_TOKENS and not RUN_NAME.endswith('_cot'):
    RUN_NAME = f'{RUN_NAME}_cot'
RUN_DIR = RUN_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DATASET_PATH = RUN_DIR / 'generate_data' / 'dynamic_niah_v2.jsonl'

# Final archive destination directory. A `{RUN_NAME}.zip` archive is moved here.
if RESULTS_PATH is None:
    RESULTS_PATH = Path('results/counting') / RUN_NAME

print('Run name:', RUN_NAME)
print('Runtime run dir:', RUN_DIR)
print('Generated dataset path:', GENERATED_DATASET_PATH)
print('Final archive destination:', RESULTS_PATH)
print('Hidden-state layers:', LAYERS)
print('PCA test count:', PCA_TEST_COUNT)
print('Selected ablation examples:', SELECT_EXAMPLE_ID)
print('Use thinking:', USE_THINKING)
print('Analyze reasoning tokens:', ANALYZE_REASONING_TOKENS)
print('CoT ablation max new tokens:', MAX_NEW_TOKENS_FOR_COT)


Run name: run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400
Runtime run dir: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400
Generated dataset path: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/generate_data/dynamic_niah_v2.jsonl
Final archive destination: results/counting/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400
Hidden-state layers: [4, 8, 12, 16, 20, 24, 28]
PCA test count: 10
Selected ablation examples: [1, 5, 9]


## 3. Optional haystack preparation

If the Paul Graham haystack already has more than 10 text files of at least 5 KB, this step is skipped.

In [4]:
haystack_dir = REPO_DIR / 'data/haystacks/paul_graham'
ready_files = [p for p in haystack_dir.glob('*.txt') if p.stat().st_size >= 5 * 1024] if haystack_dir.exists() else []
if len(ready_files) > 10:
    print(f'Haystack ready: {len(ready_files)} sufficiently large files.')
else:
    cmd = ['python', 'scripts/gather_paul_graham_essays_v2.py', '--out-dir', 'data/haystacks/paul_graham']
    print('Preparing haystack:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_DIR, env={**os.environ, 'PYTHONPATH': 'src'})


Haystack ready: 33 sufficiently large files.


## 4. Generate the dataset

In [5]:
config = {
    'task_type': TASK_TYPE,
    'tokenizer_name': TOKENIZER_NAME,
    'num_examples': NUM_EXAMPLES,
    'target_haystack_tokens': TARGET_HAYSTACK_TOKENS,
    'num_needles': NUM_NEEDLES,
    'insertion_positions': INSERTION_POSITIONS,
    'prompt_style': PROMPT_STYLE,
    'global_random_seed': GLOBAL_RANDOM_SEED,
    'haystack_seed': HAYSTACK_SEED,
    'needle_seed': NEEDLE_SEED,
    'control_switch': CONTROL_SWITCH,
    'thinking_mode': USE_THINKING,
    'analyze_reasoning_tokens': ANALYZE_REASONING_TOKENS,
    'max_new_tokens_for_cot': MAX_NEW_TOKENS_FOR_COT,
    'temperature': 0.0,
    'layers': LAYERS,
    # Match the single-example notebook: write all runtime artifacts locally.
    'results_root': str(RUN_ROOT),
    'run_name': RUN_NAME,
    'cache_dir': str(HF_CACHE_DIR),
}
config_path = RUN_DIR / 'counting_config.json'
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(config_path.read_text())

cmd = ['python', '-u', 'scripts/generate_dynamic_niah_v2.py', '--config', str(config_path)]
run_streamed(cmd)
if not GENERATED_DATASET_PATH.exists():
    raise FileNotFoundError(f'Expected generated dataset was not written: {GENERATED_DATASET_PATH}')
print('Generated dataset ready:', GENERATED_DATASET_PATH)


{
  "task_type": "match_count",
  "tokenizer_name": "Qwen/Qwen3-8B",
  "num_examples": 20,
  "target_haystack_tokens": 1000,
  "num_needles": 3,
  "insertion_positions": [
    100,
    200,
    400
  ],
  "prompt_style": "easier",
  "global_random_seed": 42,
  "haystack_seed": 123,
  "needle_seed": 456,
  "control_switch": [
    true,
    false,
    false
  ],
  "layers": [
    4,
    8,
    12,
    16,
    20,
    24,
    28
  ],
  "results_root": "/content",
  "run_name": "run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400",
  "cache_dir": "/content/drive/MyDrive/Colab Notebooks/huggingface_models"
}
$ python -u scripts/generate_dynamic_niah_v2.py --config /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/counting_config.json
/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v10/scripts/generate_dynamic_niah_v2.py:117: UserWarning: Ignoring config keys that are not DynamicNiahV2Config fields: layers
  **dy

## 5. Optional response generation and metrics

In [6]:
if RUN_GENERATION_EVAL:
    print('Starting response generation/eval...', flush=True)
    eval_config = dict(config)
    eval_config['thinking_mode'] = USE_THINKING
    eval_config['analyze_reasoning_tokens'] = ANALYZE_REASONING_TOKENS
    eval_config['max_new_tokens_for_cot'] = MAX_NEW_TOKENS_FOR_COT
    eval_config_path = RUN_DIR / 'counting_eval_config.json'
    eval_config_path.write_text(json.dumps(eval_config, indent=2), encoding='utf-8')
    print(f'Evaluation thinking mode: {USE_THINKING}')
    print(eval_config_path.read_text())
    cmd = ['python', '-u', 'scripts/gen_responses.py', '--config', str(eval_config_path), '--model', MODEL_NAME]
    run_streamed(cmd)
    print('Finished response generation/eval.', flush=True)
else:
    print('Skipped response generation/eval. Set RUN_GENERATION_EVAL=True to run it.')


Starting response generation/eval...
Evaluation thinking mode: False
{
  "task_type": "match_count",
  "tokenizer_name": "Qwen/Qwen3-8B",
  "num_examples": 20,
  "target_haystack_tokens": 1000,
  "num_needles": 3,
  "insertion_positions": [
    100,
    200,
    400
  ],
  "prompt_style": "easier",
  "global_random_seed": 42,
  "haystack_seed": 123,
  "needle_seed": 456,
  "control_switch": [
    true,
    false,
    false
  ],
  "layers": [
    4,
    8,
    12,
    16,
    20,
    24,
    28
  ],
  "results_root": "/content",
  "run_name": "run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400",
  "cache_dir": "/content/drive/MyDrive/Colab Notebooks/huggingface_models",
  "thinking_mode": false
}
$ python -u scripts/gen_responses.py --config /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/counting_eval_config.json --model Qwen/Qwen3-8B
/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v10/scripts/gen_respon

## 6. Hidden-state analysis

In [7]:
if RUN_HIDDEN_STATE_ANALYSIS:
    print('Starting hidden-state analysis...', flush=True)
    print('Hidden-state layers:', LAYERS, flush=True)
    switch_args = ['true' if x else 'false' for x in CONTROL_SWITCH]
    layer_args = [str(layer) for layer in LAYERS]
    cmd = [
        'python',
        '-u',
        'scripts/analyze_hidden_states.py',
        '--config',
        str(config_path),
        '--model',
        MODEL_NAME,
        '--layers',
        *layer_args,
        '--pca-test-count',
        str(PCA_TEST_COUNT),
        '--control_switch',
        *switch_args,
    ]
    if ANALYZE_REASONING_TOKENS:
        cmd.append('--analyze-reasoning-tokens')
    run_streamed(cmd)
    if not GENERATED_DATASET_PATH.exists():
        raise FileNotFoundError(f'Hidden-state analysis did not leave the generated dataset at: {GENERATED_DATASET_PATH}')
    print('Finished hidden-state analysis.', flush=True)
else:
    print('Skipped hidden-state analysis.')


Starting hidden-state analysis...
Hidden-state layers: [4, 8, 12, 16, 20, 24, 28]
$ python -u scripts/analyze_hidden_states.py --config /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/counting_config.json --model Qwen/Qwen3-8B --layers 4 8 12 16 20 24 28 --pca-test-count 10 --control_switch true false false
/content/drive/MyDrive/Colab Notebooks/compression/dataset-generation-main-v10/scripts/analyze_hidden_states.py:242: UserWarning: Ignoring config keys that are not DynamicNiahV2Config fields: layers
  **dynamic_niah_v2_config_kwargs(kwargs, warn_unknown=True)
[run] run_dir=/content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400
[run] logs=/content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/logs.txt
[run] metadata=/content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/run_metadata.json
[hidden-analysis] loaded existing dataset rows=20 path=/content/run_20260610_014341_

## 7. Optional ablation analysis

This section loops over `SELECT_EXAMPLE_ID` and writes each selected example under `/content/{RUN_NAME}/ablation_examples/example_id_*`. It uses the generated dataset at `GENERATED_DATASET_PATH`; run the dataset generation cell before running this block.


In [8]:
if RUN_ABLATION or RUN_REPRESENTATION_ABLATION or RUN_REPRESENTATION_RESTORE:
    if not GENERATED_DATASET_PATH.exists():
        raise FileNotFoundError(f'Run dataset generation before ablation; missing {GENERATED_DATASET_PATH}')
    ablation_results = run_counting_ablation_examples(
        run_dir=RUN_DIR,
        run_name=RUN_NAME,
        dataset_path=GENERATED_DATASET_PATH,
        config_path=config_path,
        model_name=MODEL_NAME,
        selected_example_ids=SELECT_EXAMPLE_ID,
        num_examples=NUM_EXAMPLES,
        hidden_layers=LAYERS,
        run_ablation=RUN_ABLATION,
        ablation_config_path=ABLATION_CONFIG_PATH,
        num_critical_tokens=NUM_CRITICAL_TOKENS,
        ablation_random_seed=ABLATION_RANDOM_SEED,
        critical_token_calc_layer=CRITICAL_TOKEN_CALC_LAYER,
        run_representation_ablation=RUN_REPRESENTATION_ABLATION,
        representation_config_path=ABLATION_REPRESENTATION_CONFIG_PATH,
        representation_num_critical_tokens=REPRESENTATION_NUM_CRITICAL_TOKENS,
        randomize_from_top_layer=RANDOMIZE_FROM_TOP_LAYER,
        run_representation_restore=RUN_REPRESENTATION_RESTORE,
        representation_restore_config_path=ABLATION_REPRESENTATION_RESTORE_CONFIG_PATH,
        restore_num_critical_tokens=RESTORE_NUM_CRITICAL_TOKENS,
        restore_randomize_from_top_layer=RESTORE_RANDOMIZE_FROM_TOP_LAYER,
        analyze_reasoning_tokens=ANALYZE_REASONING_TOKENS,
    )
    print('Ablation results:', ablation_results)
else:
    ablation_results = None
    print('Skipped ablation analysis. Enable RUN_ABLATION, RUN_REPRESENTATION_ABLATION, or RUN_REPRESENTATION_RESTORE to run it.')



===== Counting ablation example 1: dynamic_niah_v2_2 =====
Dataset source: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/generate_data/dynamic_niah_v2.jsonl
Saved one-row ablation dataset: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_1/generate_data/dynamic_niah_v2.jsonl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Uncontrolled input shape: (1, 1134)
Controlled input shape: (1, 1134)
Saved input metadata: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_1/generate_data/inputs_1.json
[hidden-analysis] token-length mismatch/alignment  normal_len=1134 control_len=1134 insertion_position=148 offset=0
measurements: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_1/tensors/inputs_1.pt
hidden: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_1/tensors/hidden_inputs_1.pt
figure: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_1/figures/inputs_1.png
input_ids_table: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_1/tables/model_input_ids.txt
Token-level ablation summary: {'config': {'num_critical_t

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Uncontrolled input shape: (1, 1131)
Controlled input shape: (1, 1133)
Saved input metadata: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_5/generate_data/inputs_5.json
[hidden-analysis] token-length mismatch/alignment  normal_len=1131 control_len=1133 insertion_position=147 offset=2
measurements: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_5/tensors/inputs_5.pt
hidden: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_5/tensors/hidden_inputs_5.pt
figure: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_5/figures/inputs_5.png
input_ids_table: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_5/tables/model_input_ids.txt
Token-level ablation summary: {'config': {'num_critical_t

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Uncontrolled input shape: (1, 1138)
Controlled input shape: (1, 1138)
Saved input metadata: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_9/generate_data/inputs_9.json
[hidden-analysis] token-length mismatch/alignment  normal_len=1138 control_len=1138 insertion_position=148 offset=0
measurements: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_9/tensors/inputs_9.pt
hidden: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_9/tensors/hidden_inputs_9.pt
figure: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_9/figures/inputs_9.png
input_ids_table: /content/run_20260610_014341_Qwen3-8B_match_count_easier_1000_needles_100_200_400/ablation_examples/example_id_9/tables/model_input_ids.txt
Token-level ablation summary: {'config': {'num_critical_t

## 8. Clean large tensors and archive outputs

This mirrors the single-example notebook: clean oversized intermediates from the local `/content/{RUN_NAME}` run folder, remove Q/K caches, attention-stat input folders, and corruption JSONL logs, create one local zip, then move that single archive to `RESULTS_PATH` on Drive.


In [ ]:
# Final archive cleanup: these intermediate artifacts are useful during analysis but
# very large in Colab runs, so delete them from /content/{RUN_NAME} before zipping.
cleanup_report = cleanup_counting_archive_artifacts(
    RUN_DIR,
    delete_large_pt=DELETE_LARGE_PT_WHEN_DONE,
    max_pt_bytes=100 * 1024 * 1024,
)
print(format_counting_cleanup_summary(cleanup_report))
if not DELETE_LARGE_PT_WHEN_DONE:
    print('Large .pt deletion disabled; qk_cache, attention_stats/input_*, and corrupted_needle_tokens.jsonl were still removed.')

archive_path = archive_directory(RUN_DIR, RESULTS_PATH, archive_name=RUN_NAME)
print('Archive written to:', archive_path)


## 9. Optional: release Colab runtime


In [ ]:
RELEASE_RUNTIME = True
if RELEASE_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print('Runtime retained. Set RELEASE_RUNTIME=True and rerun this cell to release it.')


Runtime retained. Set RELEASE_RUNTIME=True and rerun this cell to release it.
